<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN/blob/main/07_rag/QA_practice_with_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering: Extractive and Abstractive

An extractive model selects a span from a context. An abstractive model generates an answer token by token. Neither one searches for evidence unless we connect it to a retriever.

Let's inspect both approaches, when they work and what happens when the answer is missing.

In [1]:
# @title Setup
%pip install -q "transformers==4.55.4" "sentencepiece==0.2.0" "matplotlib==3.10.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Extractive QA: model and context

We will kick off with a basic QA system based on DistilBERT, a lighter version of BERT, which provides a balance between performance and resource utilization. This system will answer questions based on extracting information from a provided context.

The model below was fine-tuned for extractive QA. We will keep the context fixed so every result is reproducible.

In [2]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model_id = "distilbert/distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)
model.eval()

context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in "
    "Paris, France. It is named after the engineer Gustave Eiffel, whose company "
    "designed and built the tower."
)

print(f"Model: {model_id}")
print(f"Context: {context}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Model: distilbert/distilbert-base-cased-distilled-squad
Context: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.


## Selecting an answer span

The model assigns a start logit and an end logit to every token. We search for the highest-scoring valid pair inside the context, with the end after the start and a maximum answer length of 20 tokens.

The reported score is the normalized score of the selected span among the valid spans. It is useful for comparing candidates in this example, but it is not a calibrated probability that the answer is correct.

In [3]:
# @title Extractive QA function
def extract_answer(question, context, max_answer_length=20):
    encoded = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        return_offsets_mapping=True,
    )
    sequence_ids = encoded.sequence_ids(0)
    offsets = encoded.pop("offset_mapping")[0]

    with torch.no_grad():
        output = model(**encoded)

    start_logits = output.start_logits[0]
    end_logits = output.end_logits[0]
    token_count = len(start_logits)
    positions = torch.arange(token_count)
    context_mask = torch.tensor([sequence_id == 1 for sequence_id in sequence_ids])

    span_scores = start_logits[:, None] + end_logits[None, :]
    valid_spans = (
        (positions[:, None] <= positions[None, :])
        & ((positions[None, :] - positions[:, None]) < max_answer_length)
        & context_mask[:, None]
        & context_mask[None, :]
    )
    span_scores = span_scores.masked_fill(~valid_spans, -torch.inf)

    flat_index = int(torch.argmax(span_scores))
    start_index = flat_index // token_count
    end_index = flat_index % token_count
    start_character = int(offsets[start_index, 0])
    end_character = int(offsets[end_index, 1])
    answer = context[start_character:end_character]

    valid_score_distribution = torch.softmax(span_scores[valid_spans], dim=0)
    span_score = float(valid_score_distribution.max())
    return {"answer": answer, "span_score": span_score}

## Questions answered by the context

Test three questions whose answers appear explicitly in the text.

In [4]:
answerable_questions = [
    ("Who designed the Eiffel Tower?", "Gustave Eiffel"),
    ("Where is the Eiffel Tower?", "Champ de Mars in Paris, France"),
    ("What is the tower made of?", "wrought-iron"),
]

answerable_results = []
for question, expected in answerable_questions:
    result = extract_answer(question, context)
    answerable_results.append(result)
    verdict = "PASS" if result["answer"] == expected else "FAIL"
    print(
        f"{verdict} | {question:<35} answer={result['answer']!r:<38} "
        f"span_score={result['span_score']:.4f}"
    )

PASS | Who designed the Eiffel Tower?      answer='Gustave Eiffel'                       span_score=0.9900
PASS | Where is the Eiffel Tower?          answer='Champ de Mars in Paris, France'       span_score=0.3862
PASS | What is the tower made of?          answer='wrought-iron'                         span_score=0.7613


All three answers are exact spans from the context. The score changes considerably across correct answers: *Gustave Eiffel* is close to 0.99, while the longer location is around 0.39. A lower score does not automatically mean a wrong answer.

## Questions not answered by the context

Now ask for a date that is absent and for the painter of the Mona Lisa. The correct response in both cases is *I don't know*.

<div style="border: 2px solid currentColor; border-radius: 12px; padding: 28px 24px; margin: 18px 0; text-align: center;">
  <div style="font-size: 0.85em; letter-spacing: 0.08em; text-transform: uppercase; margin-bottom: 10px;">Make a prediction</div>
  <div style="font-size: 1.5em; line-height: 1.35;"><strong>If the context does not contain an answer, what will an extractive model extract?</strong></div>
</div>

In [5]:
unanswerable_questions = [
    "In what year was the Eiffel Tower completed?",
    "Who painted the Mona Lisa?",
]

unanswerable_results = []
for question in unanswerable_questions:
    result = extract_answer(question, context)
    unanswerable_results.append(result)
    print(
        f"Q: {question}\n"
        f"A: {result['answer']!r} | span_score={result['span_score']:.4f}\n"
    )

Q: In what year was the Eiffel Tower completed?
A: 'Gustave Eiffel' | span_score=0.2087

Q: Who painted the Mona Lisa?
A: 'Gustave Eiffel' | span_score=0.9686



The model returns *Gustave Eiffel* for both unsupported questions. The second wrong answer even receives a score above 0.96.

This model was trained to select an answer span. Our decoding code also forces the result to remain inside the context. Without an explicit no-answer mechanism, it will often choose the best-looking span even when no valid answer exists.

## A retrieval error becomes a QA error

Give the Eiffel Tower question an unrelated context.

In [6]:
irrelevant_context = (
    "The Pacific Ocean is the largest and deepest ocean on Earth. "
    "It extends from the Arctic Ocean to the Southern Ocean."
)
retrieval_error = extract_answer("Who designed the Eiffel Tower?", irrelevant_context)

print(f"Context: {irrelevant_context}")
print(f"Answer: {retrieval_error['answer']!r}")
print(f"Span score: {retrieval_error['span_score']:.4f}")

Context: The Pacific Ocean is the largest and deepest ocean on Earth. It extends from the Arctic Ocean to the Southern Ocean.
Answer: 'The Pacific Ocean'
Span score: 0.0461


## Abstractive QA
Now we go beyond and delve into the realm of Abstractive Question Answering, a technique that goes beyond merely extracting text snippets from the provided context. Unlike extractive QA, abstractive QA has the ability to generate answers that may not be explicitly present in the input context but are inferred or paraphrased from it. This allows for more nuanced and human-like responses, akin to how a person might summarize or rephrase information when answering a question.

The prompt asks the model to use only the supplied context and to say *Not enough information* when the answer is absent. This is an instruction, not a guarantee.

In [8]:
from transformers import AutoModelForSeq2SeqLM

generative_model_id = "google/flan-t5-small"
generative_tokenizer = AutoTokenizer.from_pretrained(generative_model_id)
generative_model = AutoModelForSeq2SeqLM.from_pretrained(generative_model_id)
generative_model.eval()

print(f"Model: {generative_model_id}")
print(f"Parameters: {sum(p.numel() for p in generative_model.parameters()):,}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-small
Parameters: 76,961,152


FLAN-T5 Small has about 77 million parameters. Unlike the extractive model, it has a decoder and is not constrained to copy a contiguous span. That extra freedom allows abstention, but it also allows unsupported generation.

In [9]:
def generate_answer(question, context):
    prompt = (
        "Answer using only the context. If the answer is missing, say "
        f"Not enough information. Context: {context} "
        f"Question: {question} Answer:"
    )
    encoded = generative_tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.inference_mode():
        output_ids = generative_model.generate(
            **encoded,
            max_new_tokens=20,
            do_sample=False,
        )
    return generative_tokenizer.decode(output_ids[0], skip_special_tokens=True)


abstractive_questions = [
    question for question, _ in answerable_questions
] + unanswerable_questions
abstractive_results = [
    generate_answer(question, context)
    for question in abstractive_questions
]

for question, answer in zip(abstractive_questions, abstractive_results):
    print(f"Q: {question}\nA: {answer}\n")

Q: Who designed the Eiffel Tower?
A: Gustave Eiffel

Q: Where is the Eiffel Tower?
A: Paris, France

Q: What is the tower made of?
A: wrought-iron

Q: In what year was the Eiffel Tower completed?
A: Not enough

Q: Who painted the Mona Lisa?
A: Not enough information



For the supported questions, FLAN-T5 returns *Gustave Eiffel*, *Paris, France* and *wrought-iron*. The location is a shorter paraphrase rather than the longer extractive span.

For the two unsupported questions it returns *Not enough* or *Not enough information*. The wording is not perfectly consistent, but it does not invent a year or a painter in this run. This is better abstention than the extractive model, which was forced to choose *Gustave Eiffel* twice.

In [10]:
abstractive_retrieval_error = generate_answer(
    "Who designed the Eiffel Tower?",
    irrelevant_context,
)
print(f"Wrong context answer: {abstractive_retrieval_error!r}")

Wrong context answer: 'Not enough information'


With the Pacific Ocean context, the abstractive model says *Not enough information* while the extractive model returned *The Pacific Ocean*. Prompted abstention helps here, but retrieval is still the upstream component that decides whether useful evidence reaches either reader.

# Takeaway

- Extractive QA selects a span; it does not retrieve the context.
- Start and end scores can identify precise answers when the evidence is present.
- A high span score is not a calibrated guarantee that the question is answerable.
- Abstractive QA can paraphrase or abstain, but it can also generate text that is not supported.
- If retrieval supplies the wrong context, the QA model can return a confident but irrelevant span.
- RAG needs both retrieval evaluation and answer-grounding checks.